# RQ5 – Federated learning of the rule extractor (Colab, T4 GPU)

Runtime → Change runtime type → **T4 GPU**. Results are written to Google Drive so a disconnect loses nothing:
re-run the training cell and finished arms are skipped.

Arms: zero-shot · local-only per bank · centralised (pooled data) · **FedAvg (SBI+HDFC+Axis)** · FedAvg leave-one-bank-out (Axis unseen).
Model: Qwen2.5-1.5B-Instruct, 4-bit, LoRA r=16. Only adapter weights are averaged; no training example leaves its bank.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/thesis_results/fl_run1'

In [ ]:
!git clone -b round7-admin-fl https://github.com/nigamreetesh84/Thesis-Coding.git /content/Thesis-Coding
%cd /content/Thesis-Coding
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Check the data (seconds, CPU)
Per bank: 180 training examples in the bank's own house style (SBI annual gross income, HDFC net monthly income, Axis "Rs X lakh and above"), plus unseen test cards, a held-out phrasing style, the thesis documents, the Axis page and the prompt-injection file.

In [ ]:
!python -m fl.run_fl data --out-dir results_fl/data

## 2. Optional quick check (Qwen2.5-0.5B, about 10-15 min)
Run this first if you want to see the whole pipeline work before the long run.

In [ ]:
# !python -m fl.run_fl train --preset quick --out-dir /content/drive/MyDrive/thesis_results/fl_quick

## 3. Full run (Qwen2.5-1.5B, about 1.5-2.5 h on a T4)
Resumable: if Colab disconnects, run this cell again.

In [ ]:
!python -m fl.run_fl train --out-dir $OUT

## 4. Report (tables + exact McNemar)

In [ ]:
!python -m fl.run_fl report --out-dir $OUT
from IPython.display import Markdown
Markdown(open(f'{OUT}/report.md').read())

## 5. Download the results folder
Upload `results.json`, `report.md` and the `predictions_*.jsonl` files back into the chat so the numbers can be checked and written into Chapter 4.

In [ ]:
!cd $(dirname $OUT) && zip -qr fl_run1.zip $(basename $OUT) -x '*/adapters/*'
from google.colab import files
files.download(f'{OUT}/../fl_run1.zip')